In [ ]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

ROLLING_WINDOW = 7          # last N days for mean & std
SERVICE_LEVEL = 0.95        # 90% = 0.90, 95% = 0.95
LEAD_TIME = 1               # days
MACHINE_CAPACITY = 500      # max production per day

# ===============================
# LOAD DATA
# ===============================

# Example: demand history file must have columns:
# Date | Part | Actual_Demand | Current_Inventory

df = pd.read_excel("demand_data.xlsx")

# Sort properly
df = df.sort_values(["Part", "Date"])

# ===============================
# CALCULATE SAFE QUANTITY
# ===============================

def calculate_production(part_df):
    
    part_df = part_df.copy()
    
    # Rolling mean (μ)
    part_df["Mean_Demand"] = (
        part_df["Actual_Demand"]
        .rolling(ROLLING_WINDOW)
        .mean()
    )
    
    # Rolling std deviation (σ)
    part_df["Std_Demand"] = (
        part_df["Actual_Demand"]
        .rolling(ROLLING_WINDOW)
        .std()
    )
    
    # Z value from service level
    Z = norm.ppf(SERVICE_LEVEL)
    
    # Target Inventory = μL + Zσ√L
    part_df["Target_Inventory"] = (
        part_df["Mean_Demand"] * LEAD_TIME +
        Z * part_df["Std_Demand"] * np.sqrt(LEAD_TIME)
    )
    
    # Production Required
    part_df["Required_Production"] = (
        part_df["Target_Inventory"] -
        part_df["Current_Inventory"]
    )
    
    # Prevent negative production
    part_df["Required_Production"] = (
        part_df["Required_Production"]
        .clip(lower=0)
    )
    
    # Apply capacity constraint
    part_df["Final_Production"] = (
        part_df["Required_Production"]
        .clip(upper=MACHINE_CAPACITY)
    )
    
    return part_df


# Apply part-wise
result = df.groupby("Part", group_keys=False).apply(calculate_production)

# ===============================
# SAVE OUTPUT
# ===============================

result.to_excel("production_plan_output.xlsx", index=False)

print("Production planning completed.")
